# Phase 2 — CLV Modelling (BG/NBD + Gamma-Gamma)

Fits probabilistic CLV models and produces 90-day and 365-day scores for every repeat customer.

**Sections**
1. Setup & data preparation
2. BG/NBD model fitting & diagnostics
3. Gamma-Gamma assumption check & fitting
4. CLV scoring & distributions
5. CLV vs actual revenue
6. CLV segment analysis
7. Summary

## 1. Setup & Data Preparation

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import plotly.express as px
import warnings
warnings.filterwarnings('ignore')

from lifetimes import BetaGeoFitter, GammaGammaFitter
from lifetimes.plotting import plot_frequency_recency_matrix, plot_probability_alive_matrix

from models.data_pipeline import (
    load_raw_data, extract_signal_before_cleaning,
    clean_data, build_master_customer_table
)
from models.clv_model import (
    prepare_clv_data, fit_bgnbd, predict_purchases,
    check_gg_assumption, fit_gamma_gamma, score_clv,
    save_models, build_clv_scores
)

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['axes.titlesize'] = 13

In [ ]:
df_raw = load_raw_data()
return_features, cancel_features = extract_signal_before_cleaning(df_raw)
df_customers, _ = clean_data(df_raw)
master = build_master_customer_table(df_customers, return_features, cancel_features)
print(f'master table: {master.shape[0]:,} customers, {master.shape[1]} features')

In [ ]:
clv_df = prepare_clv_data(master)
print(f'\nBG/NBD input stats:')
print(clv_df[['frequency_repeat', 'recency_bgnbd', 'T_bgnbd', 'monetary']].describe().round(2))